&nbsp;

# Metadata validated and modified process (manual curation)

- ヒトの熱ストレスに関連するRNA-Seqにおいて、手動で検索したときにヒットしたデータ (GEOなどでの検索クエリではヒットしていなかった)のメタデータを整理するnotebook
- また、2025年5月以降にGEOで見つかったbulk RNA-seqのデータもこちらで処理する
- 整理する際、なぜこのようなデータフレームの処理をするのかについても記載する
- Manual Curation で収集することができたエントリーはこちらで処理している
- `GSA for Human` で発見されたデータについてもこちらに記載

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
from IPython.display import display
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(400)
pl.Config.set_tbl_cols(100)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera",
    "tax_lineage",
    "bam_md5",
    "fastq_md5",
    "sra_md5"
]

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## PRJDB3999 (RNA-seq)

- ✅ DOWNLOAD (M5 Macbook Pro + yone-hdd-fantom6)
- ✅ TRIMMING (M5 Macbook Pro + yone-hdd-fantom6)
- ✅ QUANTIFICATION (M5 Macbook Pro + yone-hdd-fantom6)

&nbsp;

- validate date: 2026-09-28
- [Identification of novel heat shock-induced long non-coding RNA in human cells](https://doi.org/10.1093/jb/mvaa126)
- [PRJDB3999](https://ddbj.nig.ac.jp/search/entry/bioproject/PRJDB3999/)
- 見逃してしまっていたデータ
- `iseq` から metadata が取得できていないので、DDBJ から `JSON` ファイルを取得
- run ごとにメタデータを取得して `concat`する方式に変更 (2 run しかないため)

In [4]:
# DRR037194
DRR037194 = pl.read_json(
    "../../Data/metadata/metadata_bioproject/DRR037194.json"
).unnest(
    "properties"
).unnest(
    "RUN_SET"
).unnest(
    "RUN"
).drop(
    "accession", "alias"
).unnest(
    "EXPERIMENT_REF"
).rename(
    {
        "identifier": "run_accession",
        "accession": "experiment_accession", # this comes from EXPERIMENT_REF, not from RUN
        "refname": "experiment_alias",
        "TITLE": "experiment_title", # same text as the TITLE of DRX033429
        "dateCreated": "first_created",
        "datePublished": "first_public",
        "dateModified": "last_updated"
    }
).with_columns(
    pl.col("run_accession").alias("accession"), # create_paired_metadata uses "accession" as the run ID
    pl.col("first_created", "first_public", "last_updated").str.slice(0, 10) # keep YYYY-MM-DD like the ENA columns
)

# dbXrefs holds the linked IDs as a list of {identifier, type, url}; turn each type into a column
DRR037194_xrefs = DRR037194.select(
    "run_accession", "dbXrefs"
).explode(
    "dbXrefs"
).unnest(
    "dbXrefs"
).pivot(
    on="type",
    index="run_accession",
    values="identifier"
).rename(
    {
        "bioproject": "study_accession",
        "biosample": "sample_accession",
        "sra-sample": "secondary_sample_accession",
        "sra-study": "secondary_study_accession",
        "sra-submission": "submission_accession"
    }
).drop(
    "sra-experiment" # already present as experiment_accession
)

DRR037194 = DRR037194.join(
    DRR037194_xrefs,
    on="run_accession",
    how="left"
).select(
    # keep the column order of PRJDB3761_modified_metadata.tsv
    "run_accession",
    "experiment_title",
    "sample_accession",
    "submission_accession",
    "last_updated",
    "secondary_study_accession",
    "status",
    "experiment_alias",
    "center_name",
    "first_created",
    "experiment_accession",
    "study_accession",
    "accession",
    "secondary_sample_accession",
    "first_public"
)

display(DRR037194)

run_accession,experiment_title,sample_accession,submission_accession,last_updated,secondary_study_accession,status,experiment_alias,center_name,first_created,experiment_accession,study_accession,accession,secondary_sample_accession,first_public
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR037194""","""Illumina HiSeq 2500 sequencing of SAMD00032080""","""SAMD00032080""","""DRA003694""","""2017-09-08""","""DRP005024""","""public""","""DRX033429""","""UT_RIC""","""2015-06-12""","""DRX033429""","""PRJDB3999""","""DRR037194""","""DRS093428""","""2019-04-27"""


In [5]:
# DRR037200
DRR037200 = pl.read_json(
    "../../Data/metadata/metadata_bioproject/DRR037200.json"
).unnest(
    "properties"
).unnest(
    "RUN_SET"
).unnest(
    "RUN"
).drop(
    "accession", "alias"
).unnest(
    "EXPERIMENT_REF"
).rename(
    {
        "identifier": "run_accession",
        "accession": "experiment_accession", # this comes from EXPERIMENT_REF, not from RUN
        "refname": "experiment_alias",
        "TITLE": "experiment_title", # same text as the TITLE of DRX033435
        "dateCreated": "first_created",
        "datePublished": "first_public",
        "dateModified": "last_updated"
    }
).with_columns(
    pl.col("run_accession").alias("accession"), # create_paired_metadata uses "accession" as the run ID
    pl.col("first_created", "first_public", "last_updated").str.slice(0, 10) # keep YYYY-MM-DD like the ENA columns
)

# dbXrefs holds the linked IDs as a list of {identifier, type, url}; turn each type into a column
DRR037200_xrefs = DRR037200.select(
    "run_accession", "dbXrefs"
).explode(
    "dbXrefs"
).unnest(
    "dbXrefs"
).pivot(
    on="type",
    index="run_accession",
    values="identifier"
).rename(
    {
        "bioproject": "study_accession",
        "biosample": "sample_accession",
        "sra-sample": "secondary_sample_accession",
        "sra-study": "secondary_study_accession",
        "sra-submission": "submission_accession"
    }
).drop(
    "sra-experiment" # already present as experiment_accession
)

DRR037200 = DRR037200.join(
    DRR037200_xrefs,
    on="run_accession",
    how="left"
).select(
    # keep the column order of PRJDB3761_modified_metadata.tsv
    "run_accession",
    "experiment_title",
    "sample_accession",
    "submission_accession",
    "last_updated",
    "secondary_study_accession",
    "status",
    "experiment_alias",
    "center_name",
    "first_created",
    "experiment_accession",
    "study_accession",
    "accession",
    "secondary_sample_accession",
    "first_public"
)

display(DRR037200)

run_accession,experiment_title,sample_accession,submission_accession,last_updated,secondary_study_accession,status,experiment_alias,center_name,first_created,experiment_accession,study_accession,accession,secondary_sample_accession,first_public
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR037200""","""Illumina HiSeq 2500 sequencing of SAMD00032086""","""SAMD00032086""","""DRA003694""","""2017-09-08""","""DRP005024""","""public""","""DRX033435""","""UT_RIC""","""2015-06-12""","""DRX033435""","""PRJDB3999""","""DRR037200""","""DRS093429""","""2019-04-27"""


In [6]:
PRJDB3999 = pl.concat(
    [
        DRR037194,
        DRR037200
    ],
    how="vertical_relaxed"
).with_columns(
    pl.lit("mRNA Seq Sample Preparation Kit").alias("rna_library"),
    pl.lit("https://doi.org/10.1093/jb/mvaa126").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit("37 C").alias("recovery treatment temperature"),
    pl.lit("120 min.").alias("recovery treatment time"),
    pl.lit("A549").alias("cell_line"),
    pl.when(pl.col("run_accession") == "DRR037194")
        .then(pl.lit("HS_wt, A549, RNA-seq"))
        .otherwise(pl.lit("CTRL_wt, A549, RNA-seq"))
        .alias("sample_title"),
    pl.col("sample_accession").alias("sample_alias"), # PRJDB3761 stores the SAMD ID here
    pl.when(pl.col("run_accession") == "DRR037194")
        .then(pl.lit("Total RNA sample, HeatShock"))
        .otherwise(pl.lit("Total RNA sample, CTRL"))
        .alias("sample_description"),
    pl.lit("Illumina HiSeq 2500").alias("instrument_model"), # but, in article, it is mentioned that "Illumina HiSeq 2000" was used
    pl.lit("ILLUMINA").alias("instrument_platform"),
    pl.lit("SINGLE").alias("library_layout"),
    pl.lit("total RNA").alias("sample_molecule_ch1"),
    pl.lit("DRP005024").alias("study_alias"),
    pl.lit("TRANSCRIPTOMIC").alias("library_source")
)

PRJDB3999.write_csv(
    "../../Data/metadata/metadata_modified/PRJDB3999_modified_metadata.tsv",
    separator='\t'
)

display(PRJDB3999)

run_accession,experiment_title,sample_accession,submission_accession,last_updated,secondary_study_accession,status,experiment_alias,center_name,first_created,experiment_accession,study_accession,accession,secondary_sample_accession,first_public,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,recovery treatment temperature,recovery treatment time,cell_line,sample_title,sample_alias,sample_description,instrument_model,instrument_platform,library_layout,sample_molecule_ch1,study_alias,library_source
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR037194""","""Illumina HiSeq 2500 sequencing of SAMD00032080""","""SAMD00032080""","""DRA003694""","""2017-09-08""","""DRP005024""","""public""","""DRX033429""","""UT_RIC""","""2015-06-12""","""DRX033429""","""PRJDB3999""","""DRR037194""","""DRS093428""","""2019-04-27""","""mRNA Seq Sample Preparation Kit""","""https://doi.org/10.1093/jb/mvaa126""","""42 C""","""60 min.""","""37 C (inferred)""","""37 C""","""120 min.""","""A549""","""HS_wt, A549, RNA-seq""","""SAMD00032080""","""Total RNA sample, HeatShock""","""Illumina HiSeq 2500""","""ILLUMINA""","""SINGLE""","""total RNA""","""DRP005024""","""TRANSCRIPTOMIC"""
"""DRR037200""","""Illumina HiSeq 2500 sequencing of SAMD00032086""","""SAMD00032086""","""DRA003694""","""2017-09-08""","""DRP005024""","""public""","""DRX033435""","""UT_RIC""","""2015-06-12""","""DRX033435""","""PRJDB3999""","""DRR037200""","""DRS093429""","""2019-04-27""","""mRNA Seq Sample Preparation Kit""","""https://doi.org/10.1093/jb/mvaa126""","""42 C""","""60 min.""","""37 C (inferred)""","""37 C""","""120 min.""","""A549""","""CTRL_wt, A549, RNA-seq""","""SAMD00032086""","""Total RNA sample, CTRL""","""Illumina HiSeq 2500""","""ILLUMINA""","""SINGLE""","""total RNA""","""DRP005024""","""TRANSCRIPTOMIC"""


In [7]:
defined_pairs_prjdb3999 = [
    ("DRR037194", "DRR037200") # HS_wt vs. CTRL_wt
]

paired_metadata_df_prjdb3999 = create_paired_metadata(
    combined_df=PRJDB3999,
    defined_sample_pairs=defined_pairs_prjdb3999,
    sample_id_column_name="accession"
)

if paired_metadata_df_prjdb3999 is not None:
    paired_metadata_df_prjdb3999.write_csv(
        "../../Data/metadata/metadata_paired/PRJDB3999_paired_metadata.tsv",
        separator="\t"
    )
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_prjdb3999)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,submission_accession,last_updated,secondary_study_accession,status,experiment_alias,center_name,first_created,experiment_accession,study_accession,secondary_sample_accession,first_public,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,recovery treatment temperature,recovery treatment time,cell_line,sample_title,sample_alias,sample_description,instrument_model,instrument_platform,library_layout,sample_molecule_ch1,study_alias,library_source
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR037194""","""DRR037200""","""DRR037194 vs DRR037200""","""Illumina HiSeq 2500 sequencing of SAMD00032080 vs Illumina HiSeq 2500 sequencing of SAMD00032086""","""SAMD00032080 vs SAMD00032086""","""DRA003694""","""2017-09-08""","""DRP005024""","""public""","""DRX033429 vs DRX033435""","""UT_RIC""","""2015-06-12""","""DRX033429 vs DRX033435""","""PRJDB3999""","""DRS093428 vs DRS093429""","""2019-04-27""","""mRNA Seq Sample Preparation Kit""","""https://doi.org/10.1093/jb/mvaa126""","""42 C""","""60 min.""","""37 C (inferred)""","""37 C""","""120 min.""","""A549""","""HS_wt, A549, RNA-seq vs CTRL_wt, A549, RNA-seq""","""SAMD00032080 vs SAMD00032086""","""Total RNA sample, HeatShock vs Total RNA sample, CTRL""","""Illumina HiSeq 2500""","""ILLUMINA""","""SINGLE""","""total RNA""","""DRP005024""","""TRANSCRIPTOMIC"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE278916 (HRA014825) (RNA-seq)

- ✅ DOWNLOAD (M3 MacBook Pro)
- ✅ TRIMMING (M3 MacBook Pro)
- ✅ QUANTIFICATION (yone-hdd-fantom6)

&nbsp;

- validate date: 2026-02-23
- [De novo assembly of nuclear stress bodies rearranges and enhances NFIL3 to restrain acute inflammatory responses](https://doi.org/10.1016/j.cell.2025.05.003)
- [GSE277764](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE277764) の Nanoporeのデータに加えて、上記の論文で登録されているデータ
- GSA for human にも同じエントリーと思われるデータを発見: [HRA014825](https://ngdc.cncb.ac.cn/gsa-human/browse/HRA014825)
- おそらく多くのデータを取得したGEOのクエリで現在実行するとヒットする可能性が高いが、このnotebookに記載する
- `Cell stimulation; For heat shock (HS) treatment, Human HeLa, THP-1 cells and HUVECs were replaced with pre-heated (42°C) conditioned medium and incubated in a culture incubator with an environmental temperature at 42°C and humidified 5% CO2 for 2 h, then recovered at 37°C for 1 h.`

In [8]:
GSE278916_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE278916/GSE278916_PEP/GSE278916_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "RNA-Seq") &
    (pl.col("sample_taxid_ch1") == 9606) &
    (~pl.col("sample_title").str.contains("_SA_"))
).with_columns(
    pl.lit("Illumina TruSeq RNA sample preparation guide (specific kit/version unspecified)").alias("rna_library"),
    pl.lit("https://doi.org/10.1016/j.cell.2025.05.003").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature")
).rename(
    {
        "big_key": "experiment_accession"
    }
)

GSE278916 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE278916.metadata.tsv",
    separator='\t'
)

GSE278916_combined = GSE278916_PEP.join(
    GSE278916,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE278916_combined = GSE278916_combined[[s.name for s in GSE278916_combined if not (s.null_count() == GSE278916_combined.height)]]

GSE278916_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE278916_modified_metadata.tsv",
    separator='\t'
)

# pl.Config.set_tbl_rows(18)
# display(GSE278916_combined)

In [9]:
defined_pairs_gse278916 = [
    ("SRR30897701", "SRR30897703"), # RNA-seq_HeLa_HS_rep1 vs. RNA-seq_HeLa_UN_rep1
    ("SRR30897700", "SRR30897702"), # RNA-seq_HeLa_HS_rep2 vs. RNA-seq_HeLa_UN_rep2
    ("SRR30897695", "SRR30897696"), # RNA-seq_HUVEC_HS_rep1 vs. RNA-seq_HUVEC_UN_rep1
    ("SRR30897694", "SRR30897699"), # RNA-seq_HUVEC_HS_rep2 vs. RNA-seq_HUVEC_UN_rep2
    ("SRR30897688", "SRR30897690"), # RNA-seq_THP-1_HS_rep1 vs. RNA-seq_THP-1_UN_rep1
    ("SRR30897691", "SRR30897689")  # RNA-seq_THP-1_HS_rep2 vs. RNA-seq_THP-1_UN_rep2
]

paired_metadata_df_gse278916 = create_paired_metadata(
    combined_df=GSE278916_combined,
    defined_sample_pairs=defined_pairs_gse278916,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse278916 is not None:
    paired_metadata_df_gse278916.write_csv(
        "../../Data/metadata/metadata_paired/GSE278916_paired_metadata.tsv",
        separator="\t"
    )
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_gse278916)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,experiment_accession,cell_line,treatment,assembly,biosample,sra,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,fastq_galaxy,country,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,fastq_ftp,base_count,library_strategy,library_source,library_layout,status,fastq_aspera,tax_lineage,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR30897701""","""SRR30897703""","""rna-seq_hela_hs_rep1 vs rna-seq_hela_un_rep1""","""RNA-seq_HeLa_HS_rep1 vs RNA-seq_HeLa_UN_rep1""","""GSM8557427 vs GSM8557425""","""Public on May 26 2025""","""Oct 06 2024""","""May 26 2025""","""SRA""","""1""","""HeLa""","""Homo sapiens""","""9606""","""total RNA""","""totalRNA_HeLa_HS_rep1 vs totalRNA_HeLa_control_rep1""","""GPL24676""","""Li,,Yang""","""liyang_fudan@fudan.edu.cn""","""Institutes of Biological Sciences""","""Fudan University""","""131 Dong-An Road""","""Shanghai""","""200032""","""China""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8557427 vs GSM8557425""","""NONE""","""GSE278916""","""0""","""SRX26301193 vs SRX26301191""","""HeLa""","""HS stressed vs untreated""","""CHM13 T2T v2.0""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN44078906 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN44…","""https://www.ncbi.nlm.nih.gov/sra?term=SRX26301193 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX263011…","""Illumina TruSeq RNA sample preparation guide (specific kit/version unspecified)""","""https://doi.org/10.1016/j.cell.2025.05.003""","""42 C""","""120 min.""","""37 C (inferred)""","""SRR30897701 vs SRR30897703""","""Illumina NovaSeq 6000 sequencing: GSM8557427: RNA-seq_HeLa_HS_rep1 Homo sapiens RNA-Seq vs Illumina …","""SAMN44078906 vs SAMN44078908""","""RNA sequencing in untreated (UN), sodium arsenite (SA) and heat shock (HS) stressed HeLa,HUVEC and T…","""SRA1986818""","""cDNA""","""2025-05-28""","""4752711227;4957546654 vs 5111104644;5326875706""","""ILLUMINA""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR308/001/SRR30897701/SRR30897701_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fa…","""missing""","""Generic""","""RNA-seq_HeLa_HS_rep1 vs RNA-seq_HeLa_UN_rep1""","""59f0d04fe7124f07a028d81569aaf7ef;ac5c50fe04e2da733feae68dfd5bd2ef vs 3a553d5703db54d745147431f3fbda9…","""SRP536890""","""69922503 vs 73555914""","""RNA sequencing in untreated (UN), sodium arsenite (SA) and heat shock (HS) stressed HeLa,HUVEC and T…","""GENERATED_FILE;GENERATED_FILE""","""RNA-seq_HeLa_HS_rep1 vs RNA-seq_HeLa_UN_rep1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE191134 (PAIRED) (RNA-seq)

- ✅ DOWNLOAD (yone-hdd-fantom6)
- ✅ TRIMMING (yone-hdd-fantom6)
- ✅ QUANTIFICATION (yone-hdd-fantom6)

&nbsp;

- validate date: 2025-10-12
- [GSE191134](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE191134)
- [Reversible phase separation of HSF1 is required for an acute transcriptional response during heat shock](https://doi.org/10.1038/s41556-022-00846-7)
- HSF1の相分離について調査している論文の中に、RNA-seqのデータが含まれていたが、なぜかヒットしていなかったので手動で収集
- 熱ストレス ＋ 相分離を解消させるためのヘキサンジオール処理をしているデータは、untreated + ヘキサンジオール処理のデータがないため今回は除外する... (SRR16555315, SRR16555316)
- HSF1に変異が導入されたM3, M1のデータには、それぞれHSとuntreatedがあるかと思っていたが、M3はHSのみ、M1はuntreatedのみだったのでなくなく今回は除外する... (SRR16555317, SRR16555318 ,SRR16555319. SRR16555320)
- 他のデータは収集、解析

In [10]:
GSE191134_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE191134/GSE191134_PEP/GSE191134_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("VAHTS universal V8 RNA-seq library prep kit for Illumina (Vazyme, cat. no. NR605) and KAPA HyperPrep kit (Roche, cat. no. KR0961)").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1038/s41556-022-00846-7").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("unknown").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE191134 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE191134.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE191134_filtered = GSE191134[[s.name for s in GSE191134 if not (s.null_count() == GSE191134.height)]]

GSE191134_combined = GSE191134_PEP.join(
    GSE191134_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left"
).sort(
    [
        "gsm_id"
    ],
    descending=False
)

GSE191134_combined = GSE191134_combined[[s.name for s in GSE191134_combined if not (s.null_count() == GSE191134_combined.height)]]

GSE191134_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE191134_modified_metadata.tsv",
    separator='\t'
)

display(GSE191134_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,genotype,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,submission_accession,library_selection,last_updated,sra_bytes,isolate,fastq_bytes,instrument_platform,sex,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,experiment_alias,center_name,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,i64,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str
"""m1-nhs-rep1""","""M1-NHS-Rep1""","""GSM5739823""","""Public on Dec 19 2021""","""Dec 17 2021""","""Dec 21 2021""","""SRA""",1,"""HeLa cells""","""Homo sapiens""",9606,"""total RNA""","""GPL24676""","""Hongchen,,Zhang""","""zhanghongchen@pku.edu.cn""",18510211543,"""Peking University""","""Beijing""","""Please Select""",100871,"""China""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5739823""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5739nnn/GSM5739823/suppl/GSM5739823_raw_count_M1_NHS_Rep1.…","""GSE191134""",0,"""SRX12757451""","""HeLa cells""","""Cancer Cell Line HeLa S6""","""untreated""","""HSF1 KO and transfected with HSF1 M1""","""GRCh38""","""raw read count""","""SAMN22550559""","""VAHTS universal V8 RNA-seq library prep kit for Illumina (Vazyme, cat. no. NR605) and KAPA HyperPrep…","""https://doi.org/10.1038/s41556-022-00846-7""","""42 C""","""60 min.""","""unknown""","""SRR16555320""","""Illumina NovaSeq 6000 sequencing: RNA-Seq of HeLa cells""","""SRA1316197""","""PCR""","""2022-03-10""",2509326542,"""HeLa cell line""","""1796276308;1815041427""","""ILLUMINA""","""female""","""Human""","""M1-NHS-biological replicate 1""","""SRP342933""",27624541,"""RNA sequencing of HeLa cell under heat shock""","""M1-NHS-biological replicate 1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: RNA-Seq of HeLa cells""",8287362300,"""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""M1-NHS-Rep1""","""M1-NHS-Rep1_R1.fq.gz""","""RNA_seq5""","""SUB10562654""","""HeLa S6""","""RNA_seq5""","""env_tax;env_tax:terrestrial""","""2022-03-10""","""Homo sapiens""",9606,"""PRJNA774101""","""SRR16555320""","""SRS10704767""","""2022-03-10""","""PRJNA774101""","""cervical cancer"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""wt-nhs-rep2""","""WT-NHS-Rep2""","""GSM5739832""","""Public on Dec 19 2021""","""Dec 17 2021""","""Dec 21 2021""","""SRA""",1,"""HeLa cells""","""Homo sapiens""",9606,"""total RNA""","""GPL24676""","""Hongchen,,Zhang""","""zhanghongchen@pku.edu.cn""",18510211543,"""Peking University""","""Beijing""","""Please Select""",100871,"""China""","""Illumina NovaSeq 6000""","""cDNA""","""tr

In [11]:
defined_pairs_gse191134 = [
    ("SRR16555321", "SRR16555323"), # wt-hs-rep2 vs wt-nhs-rep2
    ("SRR16555322", "SRR16555324")  # wt-hs-rep1 vs wt-nhs-rep1
]

paired_metadata_df_gse191134 = create_paired_metadata(
    combined_df=GSE191134_combined, 
    defined_sample_pairs=defined_pairs_gse191134,
    sample_id_column_name="accession" # default value, but specified to clarify
)

# Guard against None so that write_csv is called only on a DataFrame
if paired_metadata_df_gse191134 is not None:
    paired_metadata_df_gse191134.write_csv(
        "../../Data/metadata/metadata_paired/GSE191134_paired_metadata.tsv",
        separator="\t"
    )
    display(paired_metadata_df_gse191134)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,genotype,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,submission_accession,library_selection,last_updated,sra_bytes,isolate,fastq_bytes,instrument_platform,sex,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,experiment_alias,center_name,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR16555321""","""SRR16555323""","""wt-hs-rep2 vs wt-nhs-rep2""","""WT-HS-Rep2 vs WT-NHS-Rep2""","""GSM5739830 vs GSM5739832""","""Public on Dec 19 2021""","""Dec 17 2021""","""Dec 21 2021""","""SRA""","""1""","""HeLa cells""","""Homo sapiens""","""9606""","""total RNA""","""GPL24676""","""Hongchen,,Zhang""","""zhanghongchen@pku.edu.cn""","""18510211543""","""Peking University""","""Beijing""","""Please Select""","""100871""","""China""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5739830 vs GSM5739832""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5739nnn/GSM5739830/suppl/GSM5739830_raw_count_WT_HS_Rep2.c…","""GSE191134""","""0""","""SRX12757450 vs SRX12757448""","""HeLa cells""","""Cancer Cell Line HeLa S6""","""Heat shock at 42 °C for 60 min vs untreated""","""WT cells""","""GRCh38""","""raw read count""","""SAMN22550558 vs SAMN22550556""","""VAHTS universal V8 RNA-seq library prep kit for Illumina (Vazyme, cat. no. NR605) and KAPA HyperPrep…","""https://doi.org/10.1038/s41556-022-00846-7""","""42 C""","""60 min.""","""unknown""","""SRR16555321 vs SRR16555323""","""Illumina NovaSeq 6000 sequencing: RNA-Seq of HeLa cells""","""SRA1316197""","""PCR""","""2022-03-10""","""2367171061 vs 1888810564""","""HeLa cell line""","""1690943585;1727585337 vs 1348791234;1361810755""","""ILLUMINA""","""female""","""Human""","""WT-HS-biological replicate 2 vs WT-NHS-biological replicate 2""","""SRP342933""","""25921341 vs 20788342""","""RNA sequencing of HeLa cell under heat shock""","""WT-HS-biological replicate 2 vs WT-NHS-biological replicate 2""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: RNA-Seq of HeLa cells""","""7776402300 vs 6236502600""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""WT-HS-Rep2 vs WT-NHS-Rep2""","""WT-HS-Rep2_R1.fq.gz vs WT-NHS-Rep2_R1.fq.gz""","""RNA_seq4 vs RNA_seq2""","""SUB10562654""","""HeLa S6""","""RNA_seq4 vs RNA_seq2""","""env_tax;env_tax:terrestrial""","""2022-03-10""","""Homo sapiens""","""9606""","""PRJNA774101""","""SRS10704766 vs SRS10704764""","""2022-03-10""","""PRJNA774101""","""cervical cancer"""
"""SRR16555322""","""SRR16555324""","""wt-hs-rep1 vs wt-nhs-rep1""","""WT-HS-Rep1 vs WT-NHS-Rep1""","""GSM5739829 vs GSM5739831""","""Pub

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## PRJCA005446 (HRA000922) (RNA-seq)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2026-06-26
- perplexity で検索していたところ、たまたま、論文経由からこのデータセットを発見することができた
- [Hyperthermia differentially affects specific human stem cells and their differentiated derivatives](https://doi.org/10.1007/s13238-021-00887-y) の論文のsupplementaly fileに記載してあった
- https://pmc.ncbi.nlm.nih.gov/articles/instance/9232687/bin/13238_2021_887_MOESM1_ESM.pdf
- `RNA-seq library construction and sequencing` セクション に 詳細が記載されているので手作業で追記
- このデータセットはGSA for humanからアクセスできる
- [PRJCA005446](https://ngdc.cncb.ac.cn/bioproject/browse/PRJCA005446)
- [GSA for human のHRA000922](https://ngdc.cncb.ac.cn/gsa-human/browse/HRA000922)

In [12]:
# まずはじめにExcelファイルの各ページを元に統合する
HRA000922_cell_line = pl.read_csv(
    "../../Data/metadata/metadata_gsa/HRA000922_cell_line.csv",
    separator=","
).rename(
    {
        "Accession": "Individual Accession",
        "Cell Line Name": "cell_line"
    }
).drop(
    "Ethnicity",
    "Web Url"
)

HRA000922_sample = pl.read_csv(
    "../../Data/metadata/metadata_gsa/HRA000922_sample.csv",
    separator=","
).drop(
    "Culture Collection",
    "Karyotype",
    "Phenotype",
    "Sample Name",
    "Tissue"
)

HRA000922_combined1 = HRA000922_cell_line.join(
    HRA000922_sample,
    on="Individual Accession",
    how="inner",
    coalesce=True
).rename(
    {
        "Accession": "BioSample accession"
    }
)

pl.Config.set_tbl_rows(4)
display(HRA000922_combined1)

Individual Accession,cell_line,Gender,Tissue,BioSample accession,Public Description,Sample Title,Collection Date,Biomaterial Provider
str,str,str,str,str,str,str,str,str
"""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""HRS192104""","""hESC under 37 degree at early passage in RNAseq""","""RNAseq_ESC_37""","""2019-11-13""","""IBP"""
"""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""HRS192105""","""hESC under 39 degree at early passage in RNAseq""","""RNAseq_ESC_39""","""2019-11-13""","""IBP"""
…,…,…,…,…,…,…,…,…
"""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""HRS192116""","""Neuron under 37 degree at early passage in RNAseq""","""RNAseq_Neuron_37""","""2019-10-30""","""IBP"""
"""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""HRS192117""","""Neuron under 39 degree at early passage in RNAseq""","""RNAseq_Neuron_39""","""2019-10-30""","""IBP"""


In [13]:
HRA000922_experiment = pl.read_csv(
    "../../Data/metadata/metadata_gsa/HRA000922_experiment.csv",
    separator=","
).drop(
    "Insert size (bp)",
    "Nominal size (bp)",
    "Nominal standard deviation (bp)",
    "Planned number of cycles",
    "Library name"
).rename(
    {
        "Strategy": "library_strategy",
        "Accession": "Experiment accession"
    }
).join(
    HRA000922_combined1,
    on="BioSample accession",
    how="inner",
    coalesce=True
)

pl.Config.set_tbl_rows(4)
display(HRA000922_experiment)

Experiment accession,Experiment title,BioSample name,BioSample accession,Platform,Library Construction / Experimental Design,library_strategy,Source,Selection,Layout,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,cell_line,Gender,Tissue,Public Description,Sample Title,Collection Date,Biomaterial Provider
str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str
"""HRX214074""","""RNAseq_ESC_37_rep1""","""RNAseq_ESC_37""","""HRS192104""","""Illumina HiSeq X Ten""","""Briefly, total RNAs were extracted using TRIzol Reagent, and genomic DNAs were removed using a DNA-f…","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""",150,150,"""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""hESC under 37 degree at early passage in RNAseq""","""RNAseq_ESC_37""","""2019-11-13""","""IBP"""
"""HRX214075""","""RNAseq_ESC_37_rep2""","""RNAseq_ESC_37""","""HRS192104""","""Illumina HiSeq X Ten""","""Briefly, total RNAs were extracted using TRIzol Reagent, and genomic DNAs were removed using a DNA-f…","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""",150,150,"""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""hESC under 37 degree at early passage in RNAseq""","""RNAseq_ESC_37""","""2019-11-13""","""IBP"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HRX214100""","""RNAseq_Neuron_39_rep1""","""RNAseq_Neuron_39""","""HRS192117""","""Illumina HiSeq X Ten""","""Briefly, total RNAs were extracted using TRIzol Reagent, and genomic DNAs were removed using a DNA-f…","""RNA-Seq""","""GENOMIC""","""other""","""PAIRED""",150,150,"""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""Neuron under 39 degree at early passage in RNAseq""","""RNAseq_Neuron_39""","""2019-10-30""","""IBP"""
"""HRX214101""","""RNAseq_Neuron_39_rep2""","""RNAseq_Neuron_39""","""HRS192117""","""Illumina HiSeq X Ten""","""Briefly, total RNAs were extracted using TRIzol Reagent, and genomic DNAs were removed using a DNA-f…","""RNA-Seq""","""GENOMIC""","""other""","""PAIRED""",150,150,"""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""Neuron under 39 degree at early passage in RNAseq""","""RNAseq_Neuron_39""","""2019-10-30""","""IBP"""


In [14]:
HRA000922_run = pl.read_csv(
    "../../Data/metadata/metadata_gsa/HRA000922_run.csv",
    separator=","
).drop(
    "Reference file name",
    "MD5 for reference file",
    "Assembly Name or Accession",
    "Assembly Accession URL",
    "File name 1",
    "MD5 checksum 1",
    "DownLoad1",
    "File name 2",
    "MD5 checksum 2",
    "DownLoad2"
).join(
    HRA000922_experiment,
    on="Experiment accession",
    how="inner",
    coalesce=True
).rename(
    {
        "Accession": "accession",
        "Experiment title": "sample_name",
        "BioSample accession": "secondary_sample_accession",
        "Layout": "library_layout",
        "Tissue": "cell_type",
        "Source": "library_source",
        "Public Description": "sample_description",
        "Platform": "instrument_model",
        "Experiment accession": "experiment_accession",
        "Library Construction / Experimental Design": "rna_library",
        "Run title": "sample_title",
        "Sample Title": "sample_alias",
        "Selection": "sample_library_selection",
        "Gender": "sex",
        "Collection Date": "first_public",
        "BioSample name": "experiment_alias"
    }
).with_columns(
    pl.lit("NEBNext® UltraTM Directional RNA Library Prep Kit").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1007/s13238-021-00887-y").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("39 C").alias("stress treatment temperature"),
    pl.lit("2880 min.").alias("stress treatment time"), # 48 hours
    pl.lit("37 C").alias("control treatment temperature"),
    pl.lit("ILLUMINA").alias("instrument_platform"),
    pl.lit("PRJCA005446").alias("study_accession"),
    pl.lit("PRJCA005446").alias("study_alias"),
    pl.lit("total RNA").alias("sample_molecule_ch1") # add sample_molecule_ch1 column
    
)

HRA000922_run.write_csv(
    "../../Data/metadata/metadata_modified/PRJCA005446_modified_metadata.tsv",
    separator='\t'
)

display(HRA000922_run)

accession,sample_title,experiment_accession,Run data file type,sample_name,experiment_alias,secondary_sample_accession,instrument_model,rna_library,library_strategy,library_source,sample_library_selection,library_layout,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,cell_line,sex,cell_type,sample_description,sample_alias,first_public,Biomaterial Provider,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,instrument_platform,study_accession,study_alias,sample_molecule_ch1
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""HRR255887""","""RNAseq_ESC_37_rep1""","""HRX214074""","""fastq""","""RNAseq_ESC_37_rep1""","""RNAseq_ESC_37""","""HRS192104""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""",150,150,"""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""hESC under 37 degree at early passage in RNAseq""","""RNAseq_ESC_37""","""2019-11-13""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""
"""HRR255888""","""RNAseq_ESC_37_rep2""","""HRX214075""","""fastq""","""RNAseq_ESC_37_rep2""","""RNAseq_ESC_37""","""HRS192104""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""",150,150,"""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""hESC under 37 degree at early passage in RNAseq""","""RNAseq_ESC_37""","""2019-11-13""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HRR255913""","""RNAseq_Neuron_39_rep1""","""HRX214100""","""fastq""","""RNAseq_Neuron_39_rep1""","""RNAseq_Neuron_39""","""HRS192117""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""GENOMIC""","""other""","""PAIRED""",150,150,"""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""Neuron under 39 degree at early passage in RNAseq""","""RNAseq_Neuron_39""","""2019-10-30""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""
"""HRR255914""","""RNAseq_Neuron_39_rep2""","""HRX214101""","""fastq""","""RNAseq_Neuron_39_rep2""","""RNAseq_Neuron_39""","""HRS192117""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""GENOMIC""","""other""","""PAIRED""",150,150,"""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""Neuron under 39 degree at early passage in RNAseq""","""RNAseq_Neuron_39""","""2019-10-30""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""


In [15]:
defined_pairs_hra000922 = [
    ("HRR255889", "HRR255887"), # RNAseq_ESC_39_rep1 vs RNAseq_ESC_37_rep1
    ("HRR255890", "HRR255888"), # RNAseq_ESC_39_rep2 vs RNAseq_ESC_37_rep2
    ("HRR255893", "HRR255891"), # RNAseq_MSC_39_rep1 vs RNAseq_MSC_37_rep1
    ("HRR255894", "HRR255892"), # RNAseq_MSC_39_rep2 vs RNAseq_MSC_37_rep2
    ("HRR255897", "HRR255895"), # RNAseq_NSC_39_rep1 vs RNAseq_NSC_37_rep1
    ("HRR255898", "HRR255896"), # RNAseq_NSC_39_rep2 vs RNAseq_NSC_37_rep2
    ("HRR255901", "HRR255899"), # RNAseq_EC_39_rep1 vs RNAseq_EC_37_rep1
    ("HRR255902", "HRR255900"), # RNAseq_EC_39_rep2 vs RNAseq_EC_37_rep2
    ("HRR255905", "HRR255903"), # RNAseq_VSMC_39_rep1 vs RNAseq_VSMC_37_rep1
    ("HRR255906", "HRR255904"), # RNAseq_VSMC_39_rep2 vs RNAseq_VSMC_37_rep2
    ("HRR255909", "HRR255907"), # RNAseq_CMC_39_rep1 vs RNAseq_CMC_37_rep1
    ("HRR255910", "HRR255908"), # RNAseq_CMC_39_rep2 vs RNAseq_CMC_37_rep2
    ("HRR255913", "HRR255911"), # RNAseq_Neuron_39_rep1 vs RNAseq_Neuron_37_rep1
    ("HRR255914", "HRR255912")  # RNAseq_Neuron_39_rep2 vs RNAseq_Neuron_37_rep2
]

paired_metadata_df_hra000922 = create_paired_metadata(
    combined_df=HRA000922_run,
    defined_sample_pairs=defined_pairs_hra000922,
    sample_id_column_name="accession"
)

# Guard against None so that write_csv is called only on a DataFrame
if paired_metadata_df_hra000922 is not None:
    paired_metadata_df_hra000922.write_csv(
        "../../Data/metadata/metadata_paired/prjca005446_paired_metadata.tsv",
        separator="\t"
    )
    display(paired_metadata_df_hra000922)

else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_title,experiment_accession,Run data file type,sample_name,experiment_alias,secondary_sample_accession,instrument_model,rna_library,library_strategy,library_source,sample_library_selection,library_layout,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,cell_line,sex,cell_type,sample_description,sample_alias,first_public,Biomaterial Provider,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,instrument_platform,study_accession,study_alias,sample_molecule_ch1
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""HRR255889""","""HRR255887""","""RNAseq_ESC_39_rep1 vs RNAseq_ESC_37_rep1""","""HRX214076 vs HRX214074""","""fastq""","""RNAseq_ESC_39_rep1 vs RNAseq_ESC_37_rep1""","""RNAseq_ESC_39 vs RNAseq_ESC_37""","""HRS192105 vs HRS192104""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""","""150""","""150""","""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""hESC under 39 degree at early passage in RNAseq vs hESC under 37 degree at early passage in RNAseq""","""RNAseq_ESC_39 vs RNAseq_ESC_37""","""2019-11-13""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""
"""HRR255890""","""HRR255888""","""RNAseq_ESC_39_rep2 vs RNAseq_ESC_37_rep2""","""HRX214077 vs HRX214075""","""fastq""","""RNAseq_ESC_39_rep2 vs RNAseq_ESC_37_rep2""","""RNAseq_ESC_39 vs RNAseq_ESC_37""","""HRS192105 vs HRS192104""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""","""150""","""150""","""HRI123466""","""human embryonic stem cell (hESC)""","""male""","""embryonic stem cell""","""hESC under 39 degree at early passage in RNAseq vs hESC under 37 degree at early passage in RNAseq""","""RNAseq_ESC_39 vs RNAseq_ESC_37""","""2019-11-13""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HRR255913""","""HRR255911""","""RNAseq_Neuron_39_rep1 vs RNAseq_Neuron_37_rep1""","""HRX214100 vs HRX214098""","""fastq""","""RNAseq_Neuron_39_rep1 vs RNAseq_Neuron_37_rep1""","""RNAseq_Neuron_39 vs RNAseq_Neuron_37""","""HRS192117 vs HRS192116""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""GENOMIC""","""other""","""PAIRED""","""150""","""150""","""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""Neuron under 39 degree at early passage in RNAseq vs Neuron under 37 degree at early passage in RNAs…","""RNAseq_Neuron_39 vs RNAseq_Neuron_37""","""2019-10-30""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""
"""HRR255914""","""HRR255912""","""RNAseq_Neuron_39_rep2 vs RNAseq_Neuron_37_rep2""","""HRX214101 vs HRX214099""","""fastq""","""RNAseq_Neuron_39_rep2 vs RNAseq_Neuron_37_rep2""","""RNAseq_Neuron_39 vs RNAseq_Neuron_37""","""HRS192117 vs HRS192116""","""Illumina HiSeq X Ten""","""NEBNext® UltraTM Directional RNA Library Prep Kit""","""RNA-Seq""","""GENOMIC""","""other""","""PAIRED""","""150""","""150""","""HRI123472""","""human neuron (hNeuron)""","""male""","""neuron""","""Neuron under 39 degree at early passage in RNAseq vs Neuron under 37 degree at early passage in RNAs…","""RNAseq_Neuron_39 vs RNAseq_Neuron_37""","""2019-10-30""","""IBP""","""https://doi.org/10.1007/s13238-021-00887-y""","""39 C""","""2880 min.""","""37 C""","""ILLUMINA""","""PRJCA005446""","""PRJCA005446""","""total RNA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Join dataframe

In [16]:
dfs_manual_curation: list[pl.DataFrame | None] = [
    paired_metadata_df_prjdb3999,
    paired_metadata_df_gse278916,
    paired_metadata_df_gse191134,
    paired_metadata_df_hra000922
]

manual_curation: list[pl.DataFrame] = [df for df in dfs_manual_curation if df is not None]

In [17]:
manual_curation_df = pl.concat(manual_curation, how="diagonal_relaxed").drop(
    "Run data file type",
    "fastq_md5"
)

manual_curation_df.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_manual_curation_metadata_{execute_date}.tsv",
    separator='\t'
)

display(manual_curation_df)

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,submission_accession,last_updated,secondary_study_accession,status,experiment_alias,center_name,first_created,experiment_accession,study_accession,secondary_sample_accession,first_public,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,recovery treatment temperature,recovery treatment time,cell_line,sample_title,sample_alias,sample_description,instrument_model,instrument_platform,library_layout,sample_molecule_ch1,study_alias,library_source,sample_name,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,…,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,treatment,assembly,biosample,sra,project_name,library_selection,fastq_bytes,fastq_galaxy,country,ncbi_reporting_standard,sample_description_right,read_count,study_title,fastq_file_role,sample_title_right,description,fastq_ftp,base_count,library_strategy,fastq_aspera,tax_lineage,run_alias,library_construction_protocol,collection_date,cell_line_right,library_name,tag,scientific_name,tax_id,sample_contact_phone,sample_contact_state,gse_id,cell_type,genotype,genome_build,supplementary_files_format_and_content,sra_bytes,isolate,sex,tissue_type,Read length for mate1(bp),Read length for mate 2(bp),Individual Accession,Biomaterial Provider
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR037194""","""DRR037200""","""DRR037194 vs DRR037200""","""Illumina HiSeq 2500 sequencing of SAMD00032080 vs Illumina HiSeq 2500 sequencing of SAMD00032086""","""SAMD00032080 vs SAMD00032086""","""DRA003694""","""2017-09-08""","""DRP005024""","""public""","""DRX033429 vs DRX033435""","""UT_RIC""","""2015-06-12""","""DRX033429 vs DRX033435""","""PRJDB3999""","""DRS093428 vs DRS093429""","""2019-04-27""","""mRNA Seq Sample Preparation Kit""","""https://doi.org/10.1093/jb/mvaa126""","""42 C""","""60 min.""","""37 C (inferred)""","""37 C""","""120 min.""","""A549""","""HS_wt, A549, RNA-seq vs CTRL_wt, A549, RNA-seq""","""SAMD00032080 vs SAMD00032086""","""Total RNA sample, HeatShock vs Total RNA sample, CTRL""","""Illumina HiSeq 2500""","""ILLUMINA""","""SINGLE""","""total RNA""","""DRP005024""","""TRANSCRIPTOMIC""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""SRR30897701""","""SRR30897703""","""SRR30897701 vs SRR30897703""","""Illumina NovaSeq 6000 sequencing: GSM8557427: RNA-seq_HeLa_HS_rep1 Homo sapiens RNA-Seq vs Illumina …","""SAMN44078906 vs SAMN44078908""","""SRA1986818""","""2025-05-28""","""SRP536890""","""public""","""GSM8557427_r1 vs GSM8557425_r1""",null,"""2025-05-28""","""SRX26301193 vs SRX26301191""","""PRJNA1169616""","""SRS22829912 vs SRS22829910""","""2025-05-28""","""Illumina TruSeq RNA sample preparation guide (specific kit/version unspecified)""","""https://doi.org/10.1016/j.cell.2025.05.003""","""42 C""","""120 min.""","""37 C (inferred)""",null,null,"""HeLa""","""RNA-seq_HeLa_HS_rep1 vs RNA-seq_HeLa_UN_rep1""","""GSM8557427 vs GSM8557425""","""totalRNA_HeLa_HS_rep1 